# Auslan sign-chat backend on Colab

Starts the chat API (`research/signtest/sign_chat_backend/`, branch `recognition`) on a Colab GPU and gives it a public URL for the frontend.

- **sign → text**: Uni-Sign, Arm A from `openasl_pose_only_slt.pth` fine-tuned on Auslan-Daily (Communication BLEU-4 18.03)
- **text → sign**: SignSparK fine-tuned on Auslan-Daily Communication, round 2 (hands from retrieved keyframes, σ=1 smoothing)
- **dialogue**: Qwen2.5 Instruct, sized to the GPU (7B on a 40 GB+ card, 1.5B otherwise), or Claude / an OpenAI-compatible server / echo

**Runtime**: A100 (or L4 with High-RAM). A plain T4 has the GPU memory but only 12.7 GB of system RAM, which the model loading can run out of.

**Startup cost** is mostly copying weights from Drive, once per runtime. The first runtime copies the original files (~14 GB) and section 2b writes slim copies back to Drive; every later start copies those instead.

## 1. Drive, GPU, code

In [ ]:
import os, sys, glob, json, shutil, subprocess, time
import torch
GPU_GB = torch.cuda.get_device_properties(0).total_memory / 2**30 if torch.cuda.is_available() else 0
print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                     capture_output=True, text=True).stdout.strip() or 'NO GPU - switch the runtime to a GPU')
print(f'python {sys.version.split()[0]} | torch {torch.__version__} | GPU {GPU_GB:.0f} GB | CPUs {os.cpu_count()}')
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive'
WORK = f'{DRIVE}/auslan_work'
LOCAL = '/content/signchat_local'
os.makedirs(LOCAL, exist_ok=True)

def sh(cmd, **kw):
    """Run a shell command and fail loudly (a failing `!pip` line does not stop the notebook)."""
    r = subprocess.run(cmd, shell=isinstance(cmd, str), capture_output=True, text=True, **kw)
    if r.returncode:
        print(r.stdout[-3000:], r.stderr[-3000:])
        raise RuntimeError(f'failed: {cmd}')
    return r.stdout

# The backend code: this repository's `recognition` branch. For a private repo add a Colab secret GITHUB_TOKEN.
REPO_URL = 'https://github.com/randlyoyo/FIT5120-TE38-SignLanguage.git'
BRANCH = 'recognition'
APP = '/content/app'
CODE = f'{APP}/research/signtest'            # sign_chat_backend/, unisign/, auslan_smplx/
BACKEND = f'{CODE}/sign_chat_backend'
try:
    from google.colab import userdata
    token = userdata.get('GITHUB_TOKEN')
except Exception:
    token = None
url = REPO_URL.replace('https://', f'https://{token}@') if token else REPO_URL
current = sh(['git', '-C', APP, 'rev-parse', '--abbrev-ref', 'HEAD']).strip() if os.path.isdir(f'{APP}/.git') else None
if current != BRANCH or not os.path.isdir(BACKEND):           # missing, or an old clone of another branch
    shutil.rmtree(APP, ignore_errors=True)
    sh(['git', 'clone', '-q', '--depth', '1', '-b', BRANCH, url, APP])
else:
    sh(['git', '-C', APP, 'pull', '-q'])
print('backend at', sh(['git', '-C', APP, 'log', '-1', '--format=%h %s']))

In [ ]:
# The two research repos, pinned to the commits the models were trained with
UNISIGN, UNISIGN_COMMIT = '/content/Uni-Sign', 'eed438bcb49e30405cd6ccdfcccca330c134e830'
SSK, SSK_COMMIT = '/content/SignSparK', '22a0b4ec292233c117be09a273fd8577bbbf7d8c'
for path, repo, commit in [(UNISIGN, 'https://github.com/ZechengLi19/Uni-Sign.git', UNISIGN_COMMIT),
                           (SSK, 'https://github.com/JianHe0628/SignSparK.git', SSK_COMMIT)]:
    if not os.path.isdir(f'{path}/.git'):
        sh(['git', 'clone', '-q', repo, path])
    sh(['git', '-C', path, 'checkout', '-q', commit])

t0 = time.time()
sh([sys.executable, '-m', 'pip', 'install', '-q', '-r', f'{BACKEND}/requirements.txt', 'wandb==0.21.3', 'sacrebleu'])
sh('apt-get -qq install -y ffmpeg > /dev/null')
print(f'packages ready ({time.time() - t0:.0f}s)')

from huggingface_hub import snapshot_download
MT5 = f'{UNISIGN}/pretrained_weight/mt5-base'
snapshot_download('google/mt5-base', revision='2eb15465c5dd7f72a8f7984306ad05ebc3dd1e1f', local_dir=MT5,
                  allow_patterns=['*.json', '*.model', 'pytorch_model.bin'])
print('repos and packages ready')

## 2. Weights and data from Drive

Copied to local disk with 8 parallel reads (Drive is a network disk; parallel reads are 2-3x faster than one). Files already copied in this runtime are skipped by size.

If `auslan_work/signchat_assets/` exists (made by section 2b), the slim SignSparK weights and the compact retrieval bank are copied instead of the originals: same model, same outputs, much less to copy.

In [ ]:
from concurrent.futures import ThreadPoolExecutor
UNISIGN_RUN = 'arm_a__openasl_pose_only_slt__official_stage3__single_a100__bf16'
ASSETS = f'{WORK}/signchat_assets'                          # section 2b's output
SLIM = all(os.path.exists(p) for p in [f'{ASSETS}/bank_compact.npz', f'{ASSETS}/report.json']
           + [f'{ASSETS}/signspark/{s}.{e}' for s in ('hand', 'body', 'face') for e in ('pt', 'dropped.json')])
SMPLX_SRC = (glob.glob(f'{DRIVE}/smplx_models/**/SMPLX_NEUTRAL_2020.npz', recursive=True) or [None])[0]
jobs = {'unisign.pt': f'{WORK}/runs/{UNISIGN_RUN}/checkpoint.pt', 'SMPLX_NEUTRAL_2020.npz': SMPLX_SRC}
if SLIM:
    WEIGHTS_LOCAL, BANK_LOCAL = f'{LOCAL}/signspark_slim', f'{LOCAL}/bank_compact.npz'
    jobs['bank_compact.npz'] = f'{ASSETS}/bank_compact.npz'
    jobs.update({f'signspark_slim/{s}.{e}': f'{ASSETS}/signspark/{s}.{e}' for s in ('hand', 'body', 'face') for e in ('pt', 'dropped.json')})
else:
    WEIGHTS_LOCAL, BANK_LOCAL = f'{LOCAL}/signspark_ft_smooth', f'{LOCAL}/bank/AuslanDaily_train.lmdb'
    jobs.update({f'signspark_ft_smooth/{s}.pt': f'{WORK}/signspark_ft_smooth/final/{s}.pt' for s in ('hand', 'body', 'face')})
    bank_drive = f'{WORK}/smplx_full/lmdb_smooth/train/AuslanDaily_train.lmdb'
    jobs.update({f'bank/AuslanDaily_train.lmdb/{f}': f'{bank_drive}/{f}' for f in (os.listdir(bank_drive) if os.path.isdir(bank_drive) else [])})
missing = [k for k, v in jobs.items() if not v or not os.path.exists(v)]
if missing or (not SLIM and not any(k.startswith('bank/') for k in jobs)):
    raise SystemExit(f'missing on Drive: {missing or ["the retrieval bank LMDB"]}')
print('using', 'SLIM assets from ' + ASSETS if SLIM else 'original files (run section 2b once to make slim ones)')

def copy(item):
    dst, src = item
    out = f'{LOCAL}/{dst}'
    if os.path.exists(out) and os.path.getsize(out) == os.path.getsize(src):
        return None
    os.makedirs(os.path.dirname(out), exist_ok=True)
    t = time.time()
    shutil.copyfile(src, out + '.part')
    os.replace(out + '.part', out)
    mb, dt = os.path.getsize(out) / 2**20, max(time.time() - t, 1e-3)
    return f'  {dst}: {mb:,.0f} MB in {dt:.0f}s ({mb / dt:.0f} MB/s)'

t0 = time.time()
with ThreadPoolExecutor(8) as ex:
    for msg in ex.map(copy, sorted(jobs.items(), key=lambda kv: -os.path.getsize(kv[1]))):   # largest first
        if msg:
            print(msg, flush=True)
print(f'weights ready ({time.time() - t0:.0f}s): {sh(["du", "-sh", LOCAL]).split()[0]} in {LOCAL}')

## 2b. Once: write slim copies back to Drive

Run once, in a runtime where section 2 copied the **original** files. It reads those local copies (fast), and writes `auslan_work/signchat_assets/` to Drive; every later start then copies that instead. Nothing about the model changes, and the script checks it:

- **SignSparK weights**: a tensor is dropped only when building the model from scratch already produces it bit for bit, whatever the random seed (pretrained parts loaded at build time). Check: fresh model + original file == fresh model + slim file, every tensor.
- **Retrieval bank**: only the keyframe rows generation reads, plus each sentence's M-CLIP embedding (saves encoding 10k sentences at every start). Check: every clip's keyframe batch rebuilt from both banks, identical.

Takes roughly 10-20 minutes. Skipped when the slim assets already exist or when section 2 used them.

In [ ]:
MAKE_SLIM_ASSETS = True
if SLIM:
    print('already using slim assets:', ASSETS)
elif not MAKE_SLIM_ASSETS:
    print('skipped (MAKE_SLIM_ASSETS = False)')
else:
    tmp = f'{LOCAL}/slim_out'                          # written locally, then copied to Drive in one go
    t0 = time.time()
    p = subprocess.run([sys.executable, f'{BACKEND}/scripts/slim_assets.py', '--signspark', SSK, '--code', f'{CODE}/auslan_smplx',
                        '--weights', WEIGHTS_LOCAL, '--bank', BANK_LOCAL, '--out', tmp],
                       env=dict(os.environ, WANDB_MODE='disabled', TOKENIZERS_PARALLELISM='false', TQDM_DISABLE='1'),
                       capture_output=True, text=True)
    print(p.stdout[-6000:])
    if p.returncode:
        print(p.stderr[-6000:])
        raise RuntimeError('slim_assets.py failed; the server still works with the original files - set MAKE_SLIM_ASSETS = False')
    shutil.copytree(tmp, ASSETS + '.part', dirs_exist_ok=True)
    if os.path.exists(ASSETS):
        shutil.rmtree(ASSETS)
    os.rename(ASSETS + '.part', ASSETS)
    print(json.dumps(json.load(open(f'{ASSETS}/report.json')), indent=1)[:4000])
    print(f'slim assets on Drive ({time.time() - t0:.0f}s): {sh(["du", "-sh", ASSETS]).split()[0]}')

## 3. Config and public URL

Sized for the GPU:

- **dialogue**: `hf` picks Qwen2.5-7B-Instruct on a 40 GB+ GPU (better replies, ~15 GB), Qwen2.5-1.5B otherwise. Or `anthropic` (Claude API; Colab secret `ANTHROPIC_API_KEY`), `openai` (an OpenAI-compatible server) or `echo` (the avatar signs back what it was told: a translation mode).
- **text encoders on the GPU** (`encoder_on_gpu`) when there are 24 GB+: +6.6 GB, and each new sentence is encoded in tens of ms instead of ~0.5 s on the CPU.
- **pose extraction** on the GPU with batches of 64.

The public URL comes from a Cloudflare quick tunnel: no account, but anyone with the URL can use the API while the runtime is up.

In [ ]:
DIALOGUE = 'hf'
DIALOGUE_MODEL = {'hf': 'Qwen/Qwen2.5-7B-Instruct' if GPU_GB >= 35 else 'Qwen/Qwen2.5-1.5B-Instruct',
                  'anthropic': 'claude-opus-5'}.get(DIALOGUE, '')
PORT = 8000

if DIALOGUE == 'anthropic':
    from google.colab import userdata
    os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')

# tunnel first, so the server knows its public URL (restarted if this cell is run again)
if 'tunnel' in globals() and tunnel.poll() is None:
    tunnel.terminate()
if not os.path.exists('/content/cloudflared'):
    sh(['wget', '-q', '-O', '/content/cloudflared',
        'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64'])
    os.chmod('/content/cloudflared', 0o755)
tunnel_log = open('/content/tunnel.log', 'w')
tunnel = subprocess.Popen(['/content/cloudflared', 'tunnel', '--no-autoupdate', '--url', f'http://localhost:{PORT}'],
                          stdout=tunnel_log, stderr=subprocess.STDOUT)
PUBLIC_URL = None
import re
for _ in range(60):
    time.sleep(1)
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', open('/content/tunnel.log').read())
    if m:
        PUBLIC_URL = m.group(0)
        break
print('public URL:', PUBLIC_URL)

import yaml
CFG = {
    'device': 'cuda', 'mock': False, 'media_dir': '/content/signchat_media', 'public_base_url': PUBLIC_URL or '',
    'sign2text': {'unisign_repo': UNISIGN, 'mt5_path': MT5, 'checkpoint': f'{LOCAL}/unisign.pt',
                  'code_dir': f'{CODE}/unisign', 'pose_device': 'cuda', 'pose_batch': 64},
    'text2sign': {'signspark_repo': SSK, 'code_dir': f'{CODE}/auslan_smplx', 'weights_dir': WEIGHTS_LOCAL,
                  'bank_lmdb': BANK_LOCAL, 'smplx_npz': f'{LOCAL}/SMPLX_NEUTRAL_2020.npz', 'encoder_on_gpu': GPU_GB >= 24},
    'dialogue': {'backend': DIALOGUE, 'model': DIALOGUE_MODEL},
}
with open('/content/signchat.yaml', 'w') as fh:
    yaml.safe_dump(CFG, fh, sort_keys=False)
print(open('/content/signchat.yaml').read())

## 4. Start the server

In [ ]:
import requests
if 'server' in globals() and server.poll() is None:          # re-running this cell restarts the server
    server.terminate(); server.wait(30)
env = dict(os.environ, SIGNCHAT_CONFIG='/content/signchat.yaml', WANDB_MODE='disabled',
           TOKENIZERS_PARALLELISM='false', TQDM_DISABLE='1')
server_log = open('/content/server.log', 'w')
server = subprocess.Popen([sys.executable, '-m', 'uvicorn', 'signchat.server:app', '--host', '0.0.0.0', '--port', str(PORT)],
                          cwd=BACKEND, env=env, stdout=server_log, stderr=subprocess.STDOUT)
t0 = time.time()
while True:
    time.sleep(5)
    if server.poll() is not None:
        print(open('/content/server.log').read()[-5000:])
        raise SystemExit('server exited')
    try:
        health = requests.get(f'http://localhost:{PORT}/api/health', timeout=2).json()
        break
    except Exception:
        print(f'loading models ... {time.time() - t0:.0f}s', end='\r')
print(f'models loaded in {time.time() - t0:.0f}s')
print(json.dumps(health, indent=2))
print(sh(['nvidia-smi', '--query-gpu=memory.used,memory.total', '--format=csv,noheader']))
print('API:', PUBLIC_URL, '| docs:', f'{PUBLIC_URL}/docs')

## 5. Try it

In [ ]:
r = requests.post(f'http://localhost:{PORT}/api/chat/text', json={'text': 'Hello, how are you?'}).json()
print(json.dumps(r, indent=2)[:2000])
from IPython.display import Video, display
video = r.get('reply', {}).get('sign', {}).get('video_url')
if video:
    display(Video('/content/signchat_media/' + video.rsplit('/', 1)[1], embed=True, width=480))

In [ ]:
# sign -> text -> reply: upload a signing video (mp4/webm) from your computer
from google.colab import files
up = files.upload()
name = next(iter(up))
with open(name, 'rb') as fh:
    r = requests.post(f'http://localhost:{PORT}/api/chat/sign', files={'video': (name, fh)},
                      data={'mirrored': 'false', 'session_id': 'colab-test'}).json()
print(json.dumps(r, indent=2)[:2000])

In [ ]:
# memory and speed, for the deployment question: 5 text turns after a warm-up
ts = []
for q in ['Good morning.', 'What is your name?', 'I am hungry, let us eat.', 'See you tomorrow.', 'Thank you very much.']:
    ts.append(requests.post(f'http://localhost:{PORT}/api/chat/text', json={'text': q}).json()['timings'])
import statistics
for k in ts[0]:
    print(f'{k:<10} mean {statistics.fmean(t[k] for t in ts):.2f}s  max {max(t[k] for t in ts):.2f}s')
print(sh(['nvidia-smi', '--query-gpu=memory.used,memory.total', '--format=csv,noheader']))
print(sh('free -g | head -2'))

In [ ]:
# server log (errors show up here)
print(open('/content/server.log').read()[-4000:])